<b><font size="5">12 - Document Clustering</font></b><br>
<font size="3">*Or, "The First Law of Geography: Everything is related to everything else, but near things are more related than distant things. (Waldo Tobler)"*</font>
<br><br>
<b>Lesson Goal: Cluster text documents using the k-means, OPTICS, and HDBSCAN algorithms.</b> <br>

<b>Information Requirement:</b>
- Can documents be grouped into meaningful clusters from their vectorised content?

<b>Lesson Outline:</b> 
- First, we will depart from the previously explored and preprocessed NER event dataset from last class and cluster its constituent documents - short news headlines - using the k-Means algorithms;
- Then, we will assess, optimise, and visualise the k-means clustering;
- Lastly, we will repeat the process using the density- and hierarchy-based OPTICS and HDBSCAN algorithms.

<b>Package requirements:</b>
*(Run `conda update` if needed)*
- threadpoolctl > 3.5.0
- sklearn >= 1.3

### <font color='#BFD72F'>Table of Contents </font> <a class="anchor" id='toc'></a> 
- [0. Prepare the NER Dataset](#P1)
- [1. Cluster documents using k-means](#P1)
    - [1.1 Test document clustering using k-means](#P11)
    - [1.2 Assess the quality of clustering](#P12)
    - [1.3 Optimize k-means clustering using the elbow method](#P13)
    - [1.4 Visualize clusters in 3D](#P14)
- [2. Cluster documents using hierarchical- and density-based algorithms](#P2)
    - [2.1 Cluster documents using OPTICS](#P21)
    - [2.2 Cluster documents using HDBSCAN](#P22)


In [2]:
%load_ext autoreload
%autoreload 2

#General-Purpose
import pandas as pd
import numpy as np
import re
import nltk
import pickle
import shutil
import os
from tqdm import tqdm

#Preprocessing
from utils import pipeline_v2b
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA, TruncatedSVD

#Vectorization
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

#Clustering
from sklearn.cluster import KMeans, HDBSCAN, OPTICS

#Evaluation
from sklearn.metrics import silhouette_score, calinski_harabasz_score, adjusted_mutual_info_score

#Visualization
import plotly.graph_objects as go
import plotly.express as px


In [3]:
import warnings
warnings.filterwarnings('ignore')

<font color='#BFD72F' size=5>0. Prepare the NER Dataset</font> <a class="anchor" id="P0"></a>
- 
  
[Back to TOC](#toc)

<b>Task 0.0.1a: Option 1</b> Repeat the Data Preparation process from TM11, albeit increasing sample size (recommended sample size: >0.05).

*Expected runtime: 10' for sample size = 1.0*

In [4]:
def ner_dataset_preparer(sample_size=0.25):
    
    # Load ner_dataset
    ner_dataset = pd.read_csv('data/ner_dataset.csv') ##encoding="cp1252
    ner_dataset = ner_dataset.drop(columns=["Sentence #","POS","Tag"])
    ner_dataset = ner_dataset.rename(columns={"Sentence":"raw_content"})
    ner_dataset = ner_dataset.sample(frac=sample_size, random_state=39)

    # Correct leading whitespace before punctuation
    preceding_whitespace_pattern = "(\s)(?=[\.\,\!\?\;\:\'])"
    ner_dataset["raw_content"] = ner_dataset["raw_content"].map(lambda content : re.sub(preceding_whitespace_pattern,"",content))

    # Create full preproc column
    full_preprocessor = pipeline_v2b.MainPipeline(lemmatized=True, custom_stopwords=["said","says","say"]).main_pipeline
    ner_dataset["preproc_content"] = ner_dataset["raw_content"].map(lambda content : full_preprocessor(content))

    # Create doc2vec preproc column
    doc2vec_preprocessor = pipeline_v2b.MainPipeline(lemmatized=False,no_stopwords=False,lowercase=False).main_pipeline
    ner_dataset["doc2vec_content"] = ner_dataset["raw_content"].map(lambda content : doc2vec_preprocessor(content))

    # Vectorise using BOW
    bow_vectorizer = CountVectorizer(ngram_range=(1,1), token_pattern=r"(?u)\b\w+\b")
    ner_dataset_bow_td_matrix = bow_vectorizer.fit_transform(ner_dataset["preproc_content"]).toarray()
    ner_dataset["bow_vector"] = ner_dataset_bow_td_matrix.tolist()

    # Vectorise using TF-IDF
    tfidf_vectorizer = TfidfVectorizer(ngram_range=(1,1), token_pattern=r"(?u)\b\w+\b")
    ner_dataset_tfidf_td_matrix = tfidf_vectorizer.fit_transform(ner_dataset["preproc_content"]).toarray()
    ner_dataset["tfidf_vector"] = ner_dataset_tfidf_td_matrix.tolist()

    # Vectorise using Doc2Vec
    d2v = Doc2Vec 
    documents = [TaggedDocument(doc, [i]) for i, doc in enumerate(ner_dataset["doc2vec_content"])]
    d2v_model = d2v(documents, vector_size=300, window=6, min_count=1, workers=4, epochs=20)
    ner_dataset["doc2vec_vector"] = [d2v_model.dv[idx].tolist() for idx in tqdm(range(len(ner_dataset)))]

    return ner_dataset


In [5]:
ner_dataset = ner_dataset_preparer(sample_size=0.01)

100%|██████████| 480/480 [00:00<00:00, 95135.90it/s]


In [6]:
#ner_dataset = ner_dataset.sample(frac=0.05, random_state=39)
ner_dataset

,raw_content,preproc_content,doc2vec_content,bow_vector,tfidf_vector,doc2vec_vector
47443,Thai officials say the country plans to instal...,thai official country plan install first tsuna...,Thai officials say the country plans to instal...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.009345650672912598, -0.04033878818154335, ..."
37232,Fifty-one people are believed to have killed i...,fifty-one people believe kill fiery bus crash ...,Fifty-one people are believed to have killed i...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.014989915303885937, 0.011546929366886616, 0..."
39016,"Soviet leaders favored the choreographer, alth...",soviet leader favor choreographer although mem...,Soviet leaders favored the choreographer altho...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.004420496989041567, -0.0007605666178278625,..."
42384,Attorney General Mukasey was formally sworn in...,attorney general mukasey formally swear wednes...,Attorney General Mukasey was formally sworn in...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0069778976030647755, -0.00631721131503582, ..."
8066,"During last month's six-nation talks, North Ko...",last month six-nation talk north korea agree a...,During last months six-nation talks North Kore...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.012859759852290154, 0.013140298426151276, -..."
...,...,...,...,...,...,...
43540,The French news agency reports that two underg...,french news agency report two underground hide...,The French news agency reports that two underg...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0010605311254039407, -0.008430596441030502,..."
41828,A statement from the office of Israeli Prime M...,statement office israeli prime minister ehud o...,A statement from the office of Israeli Prime M...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.007096472196280956, -0.006993670016527176, ..."
24837,Venezuelan Vice President Jose Vicente Rangel ...,venezuelan vice president jose vicente rangel ...,Venezuelan Vice President Jose Vicente Rangel ...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.002291398122906685, 0.0016263440484181046, ..."
27899,Another vehicle was reported to have been capt...,another vehicle report capture militant,Another vehicle was reported to have been capt...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.006137087009847164, 0.0023876100312918425, ..."


**Task 0.0.2:** Retrieve the BoW/TF-IDF term-document matrices and generate a (Doc2Vec) component-document matrix from the `ner_dataset`.

In [7]:
ner_bow_td_matrix = np.array([[component for component in doc] for doc in ner_dataset["bow_vector"]])
ner_tfidf_td_matrix = np.array([[component for component in doc] for doc in ner_dataset["tfidf_vector"]])
ner_doc2vec_td_matrix = np.array([[component for component in doc] for doc in ner_dataset["doc2vec_vector"]])

<font color='#BFD72F' size=5>1. Cluster documents using k-means</font> <a class="anchor" id="P1"></a>
- 
  
[Back to TOC](#toc)

<font color='#BFD72F' size=4>1.1 Test document clustering using k-means</font> <a class="anchor" id="P11"></a>
-

**Recall** the k-means algorithm:
- Initialise: Assign k centroids at random
- Do while centroid travel distance is above threshold:
    - Assign data points to the closest centroid
    - Compute the middle of the resulting polygons
    - Move the centroid to the middle while recording the centroid travel distance
    - Evaluate centroid travel distance against threshold

<img src="https://assets.blog.code-specialist.com/k_means_animation_6cdd31d106.gif" width=800>

<b>Task 1.1.1:</b> Use the <a href="https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html#sklearn.cluster.KMeans">sklearn implementation of k-means</a> to cluster TF-IDF document vectors. Use k=9.

In [8]:
kmeans_tfidf = KMeans(n_clusters=9, random_state=39, verbose=4).fit(ner_tfidf_td_matrix)

Initialization complete
Iteration 0, inertia 889.8341534776806.
Iteration 1, inertia 462.7057735970587.
Iteration 2, inertia 462.68414700916907.
Converged at iteration 2: strict convergence.
Initialization complete
Iteration 0, inertia 884.5646451185844.
Iteration 1, inertia 462.68930051238783.
Iteration 2, inertia 462.560965025018.
Converged at iteration 2: strict convergence.
Initialization complete
Iteration 0, inertia 883.5980676670421.
Iteration 1, inertia 462.8430347957376.
Iteration 2, inertia 462.7151693726726.
Iteration 3, inertia 462.68284769638814.
Converged at iteration 3: strict convergence.
Initialization complete
Iteration 0, inertia 883.5355663872065.
Iteration 1, inertia 462.3668532353532.
Iteration 2, inertia 462.2886375619086.
Converged at iteration 2: strict convergence.
Initialization complete
Iteration 0, inertia 882.9948641547477.
Iteration 1, inertia 463.0399235646753.
Iteration 2, inertia 462.9749756468678.
Converged at iteration 2: strict convergence.
Initiali

In [9]:
set(kmeans_tfidf.labels_)

{0, 1, 2, 3, 4, 5, 6, 7, 8}

In [10]:
ner_dataset["kmeans_tfidf_clusters"] = kmeans_tfidf.labels_.tolist()

In [11]:
ner_dataset.loc[ner_dataset["kmeans_tfidf_clusters"]==0]

,raw_content,preproc_content,doc2vec_content,bow_vector,tfidf_vector,doc2vec_vector,kmeans_tfidf_clusters
8066,"During last month's six-nation talks, North Ko...",last month six-nation talk north korea agree a...,During last months six-nation talks North Kore...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.012859759852290154, 0.013140298426151276, -...",0
44411,Yousuf had previously fought against Indian fo...,yousuf previously fight indian force kashmir g...,Yousuf had previously fought against Indian fo...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.012338352389633656, 0.024908041581511497, -...",0
30510,"It was originally seven stories, but five more...",originally seven story five add late,It was originally seven stories but five more ...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0026898335199803114, -0.0024636290036141872...",0
4067,Witnesses told police that they believe a suic...,witnesses tell police believe suicide bomber w...,Witnesses told police that they believe a suic...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0180300734937191, 0.0065153492614626884, 0....",0
6190,Kano officials later agreed to resume polio im...,kano official late agree resume polio immuniza...,Kano officials later agreed to resume polio im...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.009831319563090801, -0.005055810324847698, ...",0
5228,With the global economic downturn in late 2008...,global economic downturn late 2008 south korea...,With the global economic downturn in late 2008...,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.31198557364679685, 0.0, 0.0, 0.0, 0.0,...","[0.015112477354705334, 0.023205654695630074, -...",0
13522,Kostelic finished with a combined time of two ...,kostelic finish combine time two minute 213 se...,Kostelic finished with a combined time of two ...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.01743449456989765, 0.027746133506298065, 0....",0
32475,Argentina's Lionel Messi tied the match six mi...,argentinas lionel messi tie match six minute late,Argentinas Lionel Messi tied the match six min...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.006233499851077795, 0.0063720284961164, -0....",0
11885,Mr. Schrock and Mr. Grubbs later developed mor...,mr schrock mr grubbs late develop efficient st...,Mr Schrock and Mr Grubbs later developed more ...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.006637217942625284, 0.007025083992630243, -...",0
10147,It joined the Malaysian Federation in 1963 but...,join malaysian federation 1963 separate two ye...,It joined the Malaysian Federation in 1963 but...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0157918743789196, 0.03324722498655319, 0.00...",0


<b>Task 1.1.2:</b> Use the <a href="https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html#sklearn.cluster.KMeans">sklearn implementation of k-means</a> to cluster BoW document vectors. Use k=9.

In [12]:
kmeans_bow = KMeans(n_clusters=9, random_state=39).fit(ner_bow_td_matrix)
ner_dataset["kmeans_bow_clusters"] = kmeans_bow.labels_.tolist()

<b>Task 1.1.3:</b> Use the <a href="https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html#sklearn.cluster.KMeans">sklearn implementation of k-means</a> to cluster Doc2Vec document vectors. Use k=9.

In [13]:
kmeans_doc2vec = KMeans(n_clusters=9, random_state=39).fit(ner_doc2vec_td_matrix)
ner_dataset["kmeans_doc2vec_clusters"] = kmeans_doc2vec.labels_.tolist()

<font color='#BFD72F' size=4>1.2 Assess the quality of clustering</font> <a class="anchor" id="P12"></a>
-

**Understand** the <a href="https://scikit-learn.org/stable/modules/clustering.html#clustering-performance-evaluation">sklearn clustering evaluation metrics</a>:
- **Inertia (inf to 0; lower is better)**: sum of squared distances of samples to their closest cluster center *(k-means only)*;

<img src="https://scikit-learn.org/1.5/_images/sphx_glr_plot_affinity_propagation_001.png" width=600>

- **Adjusted Mutual Information (0 to 1; 1 is best)**: function that measures the agreement of the two assignments, ignoring permutations *(preferably used when there is a ground truth clustering to compare against)*;

<img src="ami.jpg" width=600>

*Note that H(U), H(V), and H(U,V) describe the entropy of the respective clusters, and that E(MI) describes the expected mutual information*


- **Silhouette Score (-1 to +1; +1 is best)**: mean silhouette coefficient of all samples, with the silhouette coefficient defined as the ratio between the difference between intercluster and intracluster distances, and the maximum distance (intra- or intercluster);

<img src="https://miro.medium.com/v2/resize:fit:1400/0*Tqd113bamewWq0_q.jpg" width=600>

- **Calinski-Harabasz Score** (0 to inf; higher is better): ratio of the sum of between-clusters dispersion and of within-cluster dispersion for all clusters (where dispersion is defined as the sum of distances squared).

<img src="https://aaanalysis.readthedocs.io/en/stable/_images/scheme_AAclust3.png" width=600>


<b>Task 1.2.1:</b> Compute the inertia, silhouette score, and Calinski-Harabasz score for all k-means clusters generated in 1.1.

In [14]:
model_list = [("kmeans tfidf k=9", kmeans_tfidf, ner_tfidf_td_matrix),
              ("kmeans BoW k=9", kmeans_bow, ner_bow_td_matrix),
              ("kmeans Doc2Vec k=9", kmeans_doc2vec, ner_doc2vec_td_matrix)]

def unsupervised_score_calculator(model_list):
    for tuple in model_list:
        #Inertia
        print("Inertia of {}: {}".format(tuple[0],tuple[1].inertia_))
        #Silhouette Score
        print("Silhouette score of {}: {}".format(tuple[0],silhouette_score(tuple[2],tuple[1].labels_)))
        #calinski-harabasz
        print("Calinski-Harabasz score of {}: {}".format(tuple[0],calinski_harabasz_score(tuple[2],tuple[1].labels_)))
        print("\n")

In [15]:
unsupervised_score_calculator(model_list)

Inertia of kmeans tfidf k=9: 462.2886375619086
Silhouette score of kmeans tfidf k=9: 0.001785619332050243
Calinski-Harabasz score of kmeans tfidf k=9: 1.6438132613555987


Inertia of kmeans BoW k=9: 6090.025516547264
Silhouette score of kmeans BoW k=9: 0.03735319637418136
Calinski-Harabasz score of kmeans BoW k=9: 2.531597314763005


Inertia of kmeans Doc2Vec k=9: 6.4389247824755245
Silhouette score of kmeans Doc2Vec k=9: 0.22000733119894753
Calinski-Harabasz score of kmeans Doc2Vec k=9: 174.57527216689053




<b>Task 1.2.2:</b> Compute the Adjusted Mutual Information between all k-means clusters generated in 1.1.

In [16]:
ami_matrix = np.array([[adjusted_mutual_info_score(tuple1[1].labels_,tuple2[1].labels_) for tuple1 in model_list]\
                        for tuple2 in model_list])

names_list = [tuples[0] for tuples in model_list]
ami_df = pd.DataFrame(data=ami_matrix,columns=names_list,index=names_list)
ami_df

,kmeans tfidf k=9,kmeans BoW k=9,kmeans Doc2Vec k=9
kmeans tfidf k=9,1.000000,0.081691,0.026334
kmeans BoW k=9,0.081691,1.000000,0.029015
kmeans Doc2Vec k=9,0.026334,0.029015,1.000000


<font color='#BFD72F' size=4>1.3 Optimize k-means clustering using the elbow method</font> <a class="anchor" id="P13"></a>
-

**Recall** that a useful heuristic to define the optimal number of clusters without incurring into overfitting is to use the "elbow method" - finding the inflection point of the intertia-cluster scree plot

<img src="https://miro.medium.com/v2/resize:fit:828/format:webp/1*-hmOKGfaUScq73e30R1q0g.png" width=600>

**Task 1.3.1:** Define a function that plots the inertia of a k-means model given a range of possible k values.

In [17]:
def inertia_plotter(tf_matrix, max_k = 10, verbose=False):
    x_k_nr = []
    y_inertia = []
    for k in tqdm(range(2,max_k+1)):
        x_k_nr.append(k)
        kmeans = KMeans(n_clusters=k,random_state=0).fit(tf_matrix)
        y_inertia.append(kmeans.inertia_)
        if verbose==True:
            print("For k = {}, inertia = {}".format(k,round(kmeans.inertia_,3)))
    fig = px.line(x=x_k_nr, y=y_inertia, markers=True)
    fig.show()

In [18]:
inertia_plotter(ner_tfidf_td_matrix,30)

100%|██████████| 29/29 [00:08<00:00,  3.30it/s]


In [19]:
inertia_plotter(ner_doc2vec_td_matrix,30)

100%|██████████| 29/29 [00:05<00:00,  5.71it/s]


**Task 1.3.2:** Define a function (`elbow_finder`) that plots that returns the optimal k value ("elbow") by selecting the sample that is furthest away from the line connecting the first (smallest k) and last (highest k) samples. Use either the <a href="https://pypi.org/project/kneed/">kneed</a> or <a href="https://pypi.org/project/shapely/">shapely</a> packages.

<img src="https://typeset.io/figures/fig-2-kneedle-algorithm-for-online-knee-detection-a-depicts-p3zcelsb.png" width=600>
</br>
<img src="https://th.bing.com/th/id/OIP.shQdO0X4pXic4oiqG_Ka5QAAAA?rs=1&pid=ImgDetMain">

In [50]:
from kneed import KneeLocator

def elbow_finder(tf_matrix, max_k=10, verbose=True):
    
    y_inertia = []
    for k in tqdm(range(1,max_k+1)):
        kmeans = KMeans(n_clusters=k,random_state=0).fit(tf_matrix)
        if verbose==True:
            print("For k = {}, inertia = {}".format(k,round(kmeans.inertia_,3)))
        y_inertia.append(kmeans.inertia_)

    x = range(1, max_k+1)
    
    kn = KneeLocator(x, y_inertia, curve='convex', direction='decreasing')
    elbow_point = kn.knee
    print(f'Optimal value of k according to the elbow method: {elbow_point}')
    
    return elbow_point


In [51]:
elbow_finder(ner_doc2vec_td_matrix,40)

  8%|▊         | 3/40 [00:00<00:02, 17.22it/s]

For k = 1, inertia = 25.532
For k = 2, inertia = 17.425
For k = 3, inertia = 12.637


 12%|█▎        | 5/40 [00:00<00:02, 12.89it/s]

For k = 4, inertia = 10.768
For k = 5, inertia = 9.019
For k = 6, inertia = 8.083


 18%|█▊        | 7/40 [00:00<00:03, 10.63it/s]

...

 98%|█████████▊| 39/40 [00:07<00:00,  3.42it/s]

For k = 39, inertia = 2.889


100%|██████████| 40/40 [00:08<00:00,  5.00it/s]

For k = 40, inertia = 2.834
Optimal value of k according to the elbow method: 7


7

In [48]:
from shapely.geometry import Point, LineString

def elbow_finder_shapely(tf_matrix, max_k=10, verbose=True):
    
    y_inertia = []
    for k in tqdm(range(1,max_k+1)):
        kmeans = KMeans(n_clusters=k,random_state=0).fit(tf_matrix)
        if verbose==True:
            print("For k = {}, inertia = {}".format(k,round(kmeans.inertia_,3)))
        y_inertia.append(kmeans.inertia_)

    # Define the line and point
    line = LineString([(1, y_inertia[0]), (max_k, y_inertia[-1])])

    elbow_point = max(range(1, max_k+1), key=lambda i: line.distance(Point(i, y_inertia[i-1])))
    print(f'Optimal value of k according to the elbow method: {elbow_point}')
    
    return elbow_point


In [52]:
elbow_finder_shapely(ner_doc2vec_td_matrix,40)

  5%|▌         | 2/40 [00:00<00:02, 12.89it/s]

For k = 1, inertia = 25.532
For k = 2, inertia = 17.425
For k = 3, inertia = 12.637


 10%|█         | 4/40 [00:00<00:02, 12.33it/s]

For k = 4, inertia = 10.768
For k = 5, inertia = 9.019


 15%|█▌        | 6/40 [00:00<00:03, 10.18it/s]

...

 98%|█████████▊| 39/40 [00:07<00:00,  3.39it/s]

For k = 39, inertia = 2.889


100%|██████████| 40/40 [00:08<00:00,  4.94it/s]

For k = 40, inertia = 2.834
Optimal value of k according to the elbow method: 7


7

**Task 1.3.3:** Use the `elbow_finder` function to select an optimal k for the TF-IDF document vectors. Then, update the `kmeans_tfidf_clusters` column of the `ner_dataset` with the new cluster labels.

In [23]:
optimal_k = elbow_finder(ner_tfidf_td_matrix,40,verbose=False)
optimal_k

100%|██████████| 40/40 [00:14<00:00,  2.76it/s]

Optimal value of k according to the elbow method: 6


6

In [24]:
kmeans_tfidf = KMeans(n_clusters=optimal_k, random_state=0).fit(ner_tfidf_td_matrix)
ner_dataset["kmeans_tfidf_clusters"] = kmeans_tfidf.labels_.tolist()

**Task 1.3.4:** As per the advice on <a href="https://scikit-learn.org/stable/auto_examples/text/plot_document_clustering.html#performing-dimensionality-reduction-using-lsa">sklearn</a> about clustering text features, use Latent Semantic Analysis (LSA) to reduce the dimensionality of the TF-IDF term-document matrix, and then use the `elbow_finder` function to select the best k. Lastly, create a `kmeans_tfidf_lsa_clusters` column of the `ner_dataset` with the new cluster labels.


In [25]:
lsa = TruncatedSVD(n_components=200) ## The number of components can also be optimized
lsa_result = lsa.fit_transform(ner_tfidf_td_matrix)
lsa_result.shape

(480, 200)

*Note: Use the explained variance ratio to understand the percentage of total variance in the dataset captured by the number of eigenvalues used in the LSA*

<img src="https://vitalflux.com/wp-content/uploads/2020/08/Screenshot-2020-08-08-at-12.05.44-PM.png">

In [26]:
lsa.explained_variance_ratio_.sum()

0.6051590054494913

In [27]:
optimal_k = elbow_finder(lsa_result,40,verbose=False)
optimal_k

100%|██████████| 40/40 [00:06<00:00,  6.19it/s]

Optimal value of k according to the elbow method: 14


14

In [28]:
kmeans_tfidf_lsa = KMeans(n_clusters=optimal_k, random_state=0).fit(lsa_result)
ner_dataset["kmeans_tfidf_lsa_clusters"] = kmeans_tfidf_lsa.labels_.tolist()

<font color='#BFD72F' size=4>1.4 Visualize clusters in 3D</font> <a class="anchor" id="P14"></a>
-

*Before visualising clusters in 3d, we want to quickly understand what they represent - to do so, we will name them using the top 5 TF-IDF tokens for each label.*

**Task 1.4.1:** Define a naming function that takes a dataset with a `preproc_content` column and the name of the column containing the labels, and that uses TF-IDF to retrieve the highest scoring N tokens for each label, concatenating them to return a topic name for each label.

In [29]:
def cluster_namer(dataset, label_column_name, nr_words=5):

    labels = list(set(dataset[label_column_name]))
    # corpus generator
    corpus = []
    for label in labels:
        label_doc = ""
        for doc in dataset["preproc_content"].loc[dataset[label_column_name]==label]:        
            label_doc = label_doc + " " + doc
        corpus.append(label_doc)
        

    tfidf_vectorizer = TfidfVectorizer(ngram_range=(1,1), token_pattern=r"(?u)\b\w+\b")
    label_name_list = []

    for idx, document in enumerate(corpus):
        corpus_tfidf_td_matrix = tfidf_vectorizer.fit_transform(corpus)
        corpus_tfidf_word_list = tfidf_vectorizer.get_feature_names_out()

        label_vocabulary = pipeline_v2b.word_freq_calculator(corpus_tfidf_td_matrix[idx].toarray(),\
                                                                        corpus_tfidf_word_list, df_output=True)
        
        label_vocabulary = label_vocabulary.head(nr_words)
        label_name = str(labels[idx])
        for jdx in range(len(label_vocabulary)):
            label_name = label_name + "_" + label_vocabulary["words"].iloc[jdx]

        label_name_list.append(label_name)

    label_name_dict = dict(zip(labels,label_name_list))
    dataset[label_column_name] = dataset[label_column_name].map(lambda label : label_name_dict[label])

    return dataset

In [30]:
ner_dataset = cluster_namer(ner_dataset, "kmeans_tfidf_clusters")
ner_dataset

,raw_content,preproc_content,doc2vec_content,bow_vector,tfidf_vector,doc2vec_vector,kmeans_tfidf_clusters,kmeans_bow_clusters,kmeans_doc2vec_clusters,kmeans_tfidf_lsa_clusters
47443,Thai officials say the country plans to instal...,thai official country plan install first tsuna...,Thai officials say the country plans to instal...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.009345650672912598, -0.04033878818154335, ...",1_kill_people_three_least_car,6,5,1
37232,Fifty-one people are believed to have killed i...,fifty-one people believe kill fiery bus crash ...,Fifty-one people are believed to have killed i...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.014989915303885937, 0.011546929366886616, 0...",1_kill_people_three_least_car,6,0,1
39016,"Soviet leaders favored the choreographer, alth...",soviet leader favor choreographer although mem...,Soviet leaders favored the choreographer altho...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.004420496989041567, -0.0007605666178278625,...",0_year_united_states_mr_two,6,0,12
42384,Attorney General Mukasey was formally sworn in...,attorney general mukasey formally swear wednes...,Attorney General Mukasey was formally sworn in...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0069778976030647755, -0.00631721131503582, ...",4_president_us_bush_venezuelan_minister,6,6,13
8066,"During last month's six-nation talks, North Ko...",last month six-nation talk north korea agree a...,During last months six-nation talks North Kore...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.012859759852290154, 0.013140298426151276, -...",2_last_week_try_officials_month,3,0,3
...,...,...,...,...,...,...,...,...,...,...
43540,The French news agency reports that two underg...,french news agency report two underground hide...,The French news agency reports that two underg...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0010605311254039407, -0.008430596441030502,...",2_last_week_try_officials_month,6,2,3
41828,A statement from the office of Israeli Prime M...,statement office israeli prime minister ehud o...,A statement from the office of Israeli Prime M...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.007096472196280956, -0.006993670016527176, ...",0_year_united_states_mr_two,6,5,10
24837,Venezuelan Vice President Jose Vicente Rangel ...,venezuelan vice president jose vicente rangel ...,Venezuelan Vice President Jose Vicente Rangel ...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.002291398122906685, 0.0016263440484181046, ...",4_president_us_bush_venezuelan_minister,6,0,7
27899,Another vehicle was reported to have been capt...,another vehicle report capture militant,Another vehicle was reported to have been capt...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.006137087009847164, 0.0023876100312918425, ...",0_year_united_states_mr_two,6,2,7


In [31]:
ner_dataset = cluster_namer(ner_dataset, "kmeans_tfidf_lsa_clusters")
ner_dataset

,raw_content,preproc_content,doc2vec_content,bow_vector,tfidf_vector,doc2vec_vector,kmeans_tfidf_clusters,kmeans_bow_clusters,kmeans_doc2vec_clusters,kmeans_tfidf_lsa_clusters
47443,Thai officials say the country plans to instal...,thai official country plan install first tsuna...,Thai officials say the country plans to instal...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-0.009345650672912598, -0.04033878818154335, ...",1_kill_people_three_least_car,6,5,1_kill_people_least_bomb_nearly
37232,Fifty-one people are believed to have killed i...,fifty-one people believe kill fiery bus crash ...,Fifty-one people are believed to have killed i...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.014989915303885937, 0.011546929366886616, 0...",1_kill_people_three_least_car,6,0,1_kill_people_least_bomb_nearly
39016,"Soviet leaders favored the choreographer, alth...",soviet leader favor choreographer although mem...,Soviet leaders favored the choreographer altho...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.004420496989041567, -0.0007605666178278625,...",0_year_united_states_mr_two,6,0,12_member_party_communist_although_leader
42384,Attorney General Mukasey was formally sworn in...,attorney general mukasey formally swear wednes...,Attorney General Mukasey was formally sworn in...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0069778976030647755, -0.00631721131503582, ...",4_president_us_bush_venezuelan_minister,6,6,13_bush_president_administration_us_prime
8066,"During last month's six-nation talks, North Ko...",last month six-nation talk north korea agree a...,During last months six-nation talks North Kore...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.012859759852290154, 0.013140298426151276, -...",2_last_week_try_officials_month,3,0,3_late_use_agree_nation_official
...,...,...,...,...,...,...,...,...,...,...
43540,The French news agency reports that two underg...,french news agency report two underground hide...,The French news agency reports that two underg...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0010605311254039407, -0.008430596441030502,...",2_last_week_try_officials_month,6,2,3_late_use_agree_nation_official
41828,A statement from the office of Israeli Prime M...,statement office israeli prime minister ehud o...,A statement from the office of Israeli Prime M...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.007096472196280956, -0.006993670016527176, ...",0_year_united_states_mr_two,6,5,10_palestinian_talk_rice_condoleezza_abbas
24837,Venezuelan Vice President Jose Vicente Rangel ...,venezuelan vice president jose vicente rangel ...,Venezuelan Vice President Jose Vicente Rangel ...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.002291398122906685, 0.0016263440484181046, ...",4_president_us_bush_venezuelan_minister,6,0,7_second_two_time_nuclear_president
27899,Another vehicle was reported to have been capt...,another vehicle report capture militant,Another vehicle was reported to have been capt...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.006137087009847164, 0.0023876100312918425, ...",0_year_united_states_mr_two,6,2,7_second_two_time_nuclear_president


**Task 1.4.2:** Define a 3d plotting function that takes a dataset, the name of a document vector column, and the name of a cluster label column, and returns a 3d graph of the document clusters.

*Note that you can define several <a href="https://plotly.com/python/discrete-color/">color scales using plotly</a> to increase readability.*

In [32]:
def plotter_3d_cluster(dataset_org, vector_column_name, cluster_label_name, write_html=False, html_name="test.html"):
    
    dataset = dataset_org.copy()
    dataset = cluster_namer(dataset, cluster_label_name)

    svd_n3 = TruncatedSVD(n_components=3)
    td_matrix = np.array([[component for component in doc] for doc in dataset[vector_column_name]])
    svd_result = svd_n3.fit_transform(td_matrix)

    for component in range(3):
        col_name = "svd_d3_x{}".format(component)
        dataset[col_name] = svd_result[:,component].tolist()

    fig = px.scatter_3d(dataset,
                        x='svd_d3_x0',
                        y='svd_d3_x1',
                        z='svd_d3_x2',
                        color=cluster_label_name,
                        title=vector_column_name+"__"+cluster_label_name,
                        opacity=0.7,
                        hover_name = "preproc_content",
                        color_discrete_sequence=px.colors.qualitative.Alphabet)

    if write_html==True:
        fig.write_html(html_name)
    fig.show()
    

In [33]:
plotter_3d_cluster(ner_dataset,"tfidf_vector","kmeans_tfidf_lsa_clusters")

In [34]:
plotter_3d_cluster(ner_dataset,"tfidf_vector","kmeans_tfidf_clusters")

<font color='#BFD72F' size=5>2. Cluster documents using hierarchical- and density-based algorithms</font> <a class="anchor" id="P2"></a>
- 
  
[Back to TOC](#toc)

**Understand** that different clustering algorithms are suited to different use cases. Read the <a href="https://scikit-learn.org/stable/modules/clustering.html#overview-of-clustering-methods">sklearn User Guide</a> to review their use cases and limitations.

<font color='#BFD72F' size=4>2.1 Cluster documents using OPTICS</font> <a class="anchor" id="P21"></a>
-

**Understand** that <a href="https://medium.com/@prasanNH/optics-hierarchical-density-based-clustering-e659e4b21764">OPTICS</a> finds core sample of high density with a minimum number of samples and expands clusters from them, keeping cluster hierarchy for a variable neighborhood radius.

<img src="reachability.jpg">

<img src="https://www.researchgate.net/publication/370354979/figure/fig4/AS:11431281154180452@1682697660013/Comparison-of-K-means-K3-and-OPTICS-on-Buckeye-corpus-cardinal-vowels.jpg" width=800>´

<b>Task 2.1.1:</b> Use the <a href="https://scikit-learn.org/stable/modules/generated/sklearn.cluster.OPTICS.html">sklearn implementation of OPTICS</a> to cluster TF-IDF document vectors. 

<i> **Note** that OPTICS can use various metrics to compute the reachability graph, including a precomputed square matrix of distances - or co-occurrences - between samples.</i>

In [35]:
optics = OPTICS(metric="minkowski",min_samples=2).fit(ner_tfidf_td_matrix)

In [36]:
silhouette_score(ner_tfidf_td_matrix, optics.labels_)

-0.009634221298488753

In [37]:
len(set(optics.labels_))

21

In [38]:
ner_dataset["optics_tfidf_clusters"] = optics.labels_.tolist()
ner_dataset.loc[ner_dataset["optics_tfidf_clusters"]==86]

,raw_content,preproc_content,doc2vec_content,bow_vector,tfidf_vector,doc2vec_vector,kmeans_tfidf_clusters,kmeans_bow_clusters,kmeans_doc2vec_clusters,kmeans_tfidf_lsa_clusters,optics_tfidf_clusters


In [39]:
plotter_3d_cluster(ner_dataset,"tfidf_vector","optics_tfidf_clusters")

<font color='#BFD72F' size=4>2.2 Cluster documents using HDBSCAN</font> <a class="anchor" id="P22"></a>
-

**Understand** that like OPTICS, <a href="https://hdbscan.readthedocs.io/en/latest/how_hdbscan_works.html">HDBSCAN</a> finds core sample of high density with a minimum number of samples and expands clusters from them over varying epsilon values, integrating the result to find a clustering that gives the best stability over epsilon, ensuring that clusters of different stabilities can be identified.

<img src="https://www.researchgate.net/profile/Solomon-Owiredu/publication/370129234/figure/fig2/AS:11431281151392423@1681978802711/A-concept-of-HDBSCAN-clustering-algorithm-a-Mutual-reachability-distance-computation.png" width=800>

<b>Task 2.1.2:</b> Use the <a href="https://scikit-learn.org/stable/modules/generated/sklearn.cluster.HDBSCAN.html#sklearn.cluster.HDBSCAN">sklearn implementation of HDBSCAN</a> to cluster TF-IDF document vectors. 

<i> **Note** that like OPTICS, HDBSCAN can use various metrics to compute the reachability graph, including a precomputed square matrix of distances - or co-occurrences - between samples.</i>

In [40]:
hdbscan = HDBSCAN(min_cluster_size=5).fit(ner_tfidf_td_matrix)
ner_dataset["hdbscan_tfidf_clusters"] = hdbscan.labels_.tolist()

In [41]:
ner_dataset.loc[ner_dataset["hdbscan_tfidf_clusters"]==0]

,raw_content,preproc_content,doc2vec_content,bow_vector,tfidf_vector,doc2vec_vector,kmeans_tfidf_clusters,kmeans_bow_clusters,kmeans_doc2vec_clusters,kmeans_tfidf_lsa_clusters,optics_tfidf_clusters,hdbscan_tfidf_clusters
37232,Fifty-one people are believed to have killed i...,fifty-one people believe kill fiery bus crash ...,Fifty-one people are believed to have killed i...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.014989915303885937, 0.011546929366886616, 0...",1_kill_people_three_least_car,6,0,1_kill_people_least_bomb_nearly,-1,0
39016,"Soviet leaders favored the choreographer, alth...",soviet leader favor choreographer although mem...,Soviet leaders favored the choreographer altho...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.004420496989041567, -0.0007605666178278625,...",0_year_united_states_mr_two,6,0,12_member_party_communist_although_leader,-1,0
8066,"During last month's six-nation talks, North Ko...",last month six-nation talk north korea agree a...,During last months six-nation talks North Kore...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.012859759852290154, 0.013140298426151276, -...",2_last_week_try_officials_month,3,0,3_late_use_agree_nation_official,3,0
12072,The military says among the dead was Taleban c...,military among dead taleban commander raz moha...,The military says among the dead was Taleban c...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.00559901911765337, -0.007779455743730068, -...",0_year_united_states_mr_two,6,5,6_community_student_several_common_include,-1,0
21667,The statement did not say where the suspect is...,statement suspect,The statement did not say where the suspect is...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0044431742280721664, 9.086916179512627e-07,...",0_year_united_states_mr_two,6,2,4_attack_police_terrorist_take_may,-1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
2385,NATO has expressed its solidarity with Britain...,nato express solidarity britain thursdays dead...,NATO has expressed its solidarity with Britain...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.005376758053898811, 0.010114963166415691, -...",2_last_week_try_officials_month,6,6,4_attack_police_terrorist_take_may,-1,0
15798,China's central Health Ministry in Beijing has...,chinas central health ministry beijing yet com...,Chinas central Health Ministry in Beijing has ...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.003370608203113079, 0.001815490541048348, -...",0_year_united_states_mr_two,6,5,7_second_two_time_nuclear_president,-1,0
43540,The French news agency reports that two underg...,french news agency report two underground hide...,The French news agency reports that two underg...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0010605311254039407, -0.008430596441030502,...",2_last_week_try_officials_month,6,2,3_late_use_agree_nation_official,9,0
41828,A statement from the office of Israeli Prime M...,statement office israeli prime minister ehud o...,A statement from the office of Israeli Prime M...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.007096472196280956, -0.006993670016527176, ...",0_year_united_states_mr_two,6,5,10_palestinian_talk_rice_condoleezza_abbas,-1,0


In [42]:
set(hdbscan.labels_)

{-1, 0, 1}

In [43]:
silhouette_score(ner_tfidf_td_matrix, hdbscan.labels_)

-0.0011226861192331925

In [44]:
plotter_3d_cluster(ner_dataset, "tfidf_vector", "hdbscan_tfidf_clusters")